# 02 · Building and preprocessing the Jilin pickup event log

Group B · Process Mining Project WS26/27 · dataset **LaDe** (Cainiao last-mile, May–Oct 2022)

This notebook follows the structure of the course tutorial *Building and Preprocessing an Event Log* (PM4Py):

```
Source table ──(Building)──▶ Event log ──(Preprocessing)──▶ Cleaned event log ──▶ Mining & Analysis
```

Feedback from the consultation on 12 Nov that this notebook answers:
1. Follow the steps of the course notebooks: building decisions, then a preprocessing pipeline with the effect of every step in numbers.
2. Discuss the case notion in terms of **convergence** and **divergence**.
3. Two activities (accept, pick up) are too simple: build a richer log from the data and explain what it shows.

Analysis question for the log: **how does a pickup unfold against its promised time window, and where does time get lost?**

In [1]:
from pathlib import Path
import contextlib, io, os, sys, tempfile, urllib.request
import pandas as pd
import pm4py
from pm4py.util import constants

constants.SHOW_PROGRESS_BAR = False
pd.set_option("display.max_colwidth", None)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW, PROC = ROOT / "data/raw", ROOT / "data/processed"
FIG, TAB = ROOT / "outputs/figures", ROOT / "outputs/tables"
for d in (RAW, PROC, FIG, TAB):
    d.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT / "src"))
from lade.viz import save_dfg_percent

# Raw data is not committed (44 MB); download it from the official Hugging Face repo if missing.
if not (RAW / "pickup_jl.csv").exists():
    urllib.request.urlretrieve("https://huggingface.co/datasets/Cainiao-AI/LaDe/resolve/main/pickup/pickup_jl.csv",
                               RAW / "pickup_jl.csv")

# Helper from the course tutorial: one summary line per log
def log_summary(df, name="log"):
    row = {"step": name, "cases": df["case:concept:name"].nunique(), "events": len(df),
           "activities": df["concept:name"].nunique(), "variants": len(pm4py.get_variants(df))}
    print(f"{name:<28} cases={row['cases']:>7,} events={row['events']:>9,} "
          f"activities={row['activities']:>2} variants={row['variants']:>5,}")
    return row



  Welcome to PM4Py — Community Version
  Open-Source License (AGPL v3)

  📚 Docs & Examples:
     https://processintelligence.solutions/pm4py

  ⚖️  License: AGPL v3 — Commercial use requires open-sourcing your application.
     Business use without open-sourcing? A commercial license is available:
     https://processintelligence.solutions/pm4py#licensing




## Part A · Building the event log

### 1. The source data
`pickup_jl.csv` has **one row per parcel** (261,801 rows for Jilin) and six timestamp columns. Timestamps have no year (the dataset covers May–Oct 2022, so we add 2022) and minute resolution; they are local wall-clock time.

| Column | Meaning | Missing |
|---|---|---|
| `accept_time` | courier accepts the pickup task in the app | never |
| `time_window_start`, `time_window_end` | the window promised to the sender | never |
| `pickup_time` | courier records the pickup | never |
| `accept_gps_time`, `pickup_gps_time` | time of the GPS fix attached to the accept / pickup | 31% / 16% |

The delivery file (`delivery_jl.csv`) is not used: it has no promised window, and its parcels cannot be linked to the pickup parcels (decision D3 in `docs/milestones.md`).

In [2]:
TS = {  # raw column -> activity, in the column order of the raw file
    "accept_time": "Accept order",
    "time_window_start": "Window opens",
    "time_window_end": "Window closes",
    "pickup_time": "Pick up order",
    "pickup_gps_time": "Pickup GPS fix",
    "accept_gps_time": "Accept GPS fix",
}
ACCEPT, PICKUP = "Accept order", "Pick up order"

parcels = pd.read_csv(RAW / "pickup_jl.csv")
for c in TS:
    parcels[c] = pd.to_datetime("2022-" + parcels[c].astype(str), format="%Y-%m-%d %H:%M:%S", errors="coerce")
parcels["courier_day"] = parcels.courier_id.astype(str) + "_" + parcels.ds.astype(str).str.zfill(4)
print(f"{len(parcels):,} parcels, {parcels.courier_id.nunique()} couriers, {parcels.courier_day.nunique():,} courier-days")
parcels[list(TS)].isna().sum().rename("missing").to_frame()

261,801 parcels, 665 couriers, 18,880 courier-days


,missing
accept_time,0
time_window_start,0
time_window_end,0
pickup_time,0
pickup_gps_time,42053
accept_gps_time,81832


### 2. Building decisions (Jans, Soffer & Jouck, 2019)

| Step | Decision | Our choice |
|---|---|---|
| 1 | Key activities and process boundaries | From the moment a courier accepts a pickup task until the end of the promised window: accept, pick up, the window, and the GPS fixes that confirm accept and pickup |
| 2 | Key tables and their relations | `pickup_jl.csv` only (one row per parcel); delivery data not linked |
| 3 | Case notion | **Parcel** (`order_id`) as the main log; courier-day as a second view (section 3) |
| 4 | Activities | **Columns as events**: each of the six timestamp columns becomes an activity |
| 5 | Attributes | Case: courier, region, AOI, AOI type, then derived status and waiting time (Enrichment). Event: `org:resource` = courier for courier actions, empty for the window (set by the platform, like *Open Incident* in the tutorial) |

### 3. Selecting the case notion: convergence and divergence

- **Case convergence**: one real event is copied into several cases, so the log contains *more events than really happened*.
- **Case divergence**: one case contains the events of several objects of the same kind, which cannot be told apart, and the model shows *false loops*.

We compare the two candidates from notebook 01 on the two original activities.

In [3]:
# Rank used whenever events share a timestamp (domain order of a pickup)
RANK = {"Accept order": 0, "Accept GPS fix": 1, "Window opens": 2, "Pick up order": 3, "Pickup GPS fix": 4, "Window closes": 5}

two = pd.concat([parcels.assign(activity=ACCEPT, timestamp=parcels.accept_time),
                 parcels.assign(activity=PICKUP, timestamp=parcels.pickup_time)], ignore_index=True)
two = two.assign(rank=two.activity.map(RANK)).sort_values(["courier_day", "timestamp", "rank"], kind="stable")

# Divergence: how many parcels share one courier-day case?
per_day = parcels.groupby("courier_day").size()
# Convergence: a courier-level event such as "Start pickup round" (first pickup after an accept),
# attached to the parcel case, would be copied into every parcel picked up in that round.
new_run = two.activity.ne(two.groupby("courier_day").activity.shift(fill_value=""))
rounds = int((new_run & (two.activity == PICKUP)).sum())

case_notions = pd.DataFrame({
    "parcel": {"cases": len(parcels), "events that really happened": len(two), "events in the log": len(two),
               "parcels per case (median / max)": "1 / 1",
               "cases with repeated Accept order": 0},
    "courier-day": {"cases": len(per_day), "events that really happened": len(two), "events in the log": len(two),
                    "parcels per case (median / max)": f"{per_day.median():.0f} / {per_day.max()}",
                    "cases with repeated Accept order": int((per_day > 1).sum())},
}).T
case_notions.to_csv(TAB / "02_case_notions.csv")
print(f"'Start pickup round': {rounds:,} real rounds, but {len(parcels):,} copies in a parcel-case log "
      f"({len(parcels) / rounds:.1f}x convergence)")
case_notions

'Start pickup round': 108,821 real rounds, but 261,801 copies in a parcel-case log (2.4x convergence)


,cases,events that really happened,events in the log,parcels per case (median / max),cases with repeated Accept order
parcel,261801,523602,523602,1 / 1,0
courier-day,18880,523602,523602,8 / 72,13490


**Decision: parcel is the main case notion.**
- The **parcel** log has neither problem: every event appears once, and every case is one parcel.
- The **courier-day** log does not converge (same number of events), but it **diverges** strongly: a courier-day mixes a median of 8 parcels, so *Accept order* repeats inside the case and we cannot tell which accept belongs to which pickup. The self-loops and the accept/pick-up loop found in notebook 01 are partly such false loops. We keep the courier-day only as a second view for batching (section 9), and say so when we show it.
- Adding courier-level events (pickup rounds) to the parcel case would cause **convergence**: each round would be copied into every parcel of the round (factor above). We therefore keep rounds out of the parcel log and analyse them on the courier level later.

The parcel log from notebook 01 had a single variant because it only used two of the six timestamp columns. Using all of them gives the parcel case real behaviour.

### 4. Columns as events

In [4]:
CASE_ATTRS = ["courier_id", "region_id", "aoi_id", "aoi_type"]
events = pd.concat([parcels[["order_id", "courier_day", *CASE_ATTRS]].assign(activity=act, timestamp=parcels[col])
                    for col, act in TS.items()], ignore_index=True)
events = events.dropna(subset=["timestamp"]).reset_index(drop=True)   # GPS fix missing -> no event
events["org:resource"] = events.courier_id.astype(str).where(~events.activity.str.startswith("Window"))
events = events.rename(columns={c: f"case:{c}" for c in CASE_ATTRS})

# Events keep the order of the raw columns when timestamps are equal ("file order"), as in the tutorial
log_raw = pm4py.format_dataframe(events, case_id="order_id", activity_key="activity", timestamp_key="timestamp")
steps = [log_summary(log_raw, "0 raw")]
save_dfg_percent(log_raw, FIG / "02_dfg_parcel_raw.png", "Parcel log, raw (file order)", unit="parcels")
log_raw[["case:concept:name", "concept:name", "time:timestamp", "org:resource", "case:aoi_type"]].head(6)

0 raw                        cases=261,801 events=1,446,921 activities= 6 variants=   65


,case:concept:name,concept:name,time:timestamp,org:resource,case:aoi_type
0,1000005,Accept GPS fix,2022-07-22 14:20:00+00:00,729,14
1,1000005,Accept order,2022-07-22 14:22:00+00:00,729,14
2,1000005,Pick up order,2022-07-22 14:50:00+00:00,729,14
3,1000005,Pickup GPS fix,2022-07-22 14:50:00+00:00,729,14
4,1000005,Window opens,2022-07-22 15:00:00+00:00,NaN,14
5,1000005,Window closes,2022-07-22 17:00:00+00:00,NaN,14


Six activities instead of two, and the parcel case now has many variants. Look at the raw DFG (`outputs/figures/02_dfg_parcel_raw.png`): with file order, *Window closes* can come before *Pick up order* although both share the minute, and GPS fixes appear on both sides of their event. We fix this in Part B.

## Part B · Preprocessing pipeline

One function per step, in the families of the lecture: Filtering, Transformation, Abstraction, Enrichment.

In [5]:
def remove_duplicates(df):
    """Filtering: the same event recorded twice."""
    return df.drop_duplicates(subset=["case:concept:name", "concept:name", "time:timestamp"])

def keep_complete(df):
    """Filtering: a parcel needs both an accept and a pickup."""
    for act in (ACCEPT, PICKUP):
        df = pm4py.filter_event_attribute_values(df, "concept:name", [act], level="case")
    return df

def drop_pickup_before_accept(df):
    """Filtering: physically impossible order (none in Jilin; the check matters for other cities)."""
    t = df[df["concept:name"].isin([ACCEPT, PICKUP])].pivot_table(
        index="case:concept:name", columns="concept:name", values="time:timestamp", aggfunc="first")
    bad = t.index[t[PICKUP] < t[ACCEPT]]
    return df[~df["case:concept:name"].isin(bad)]

def reorder(df, rank):
    """Transformation: sort by case, timestamp and a domain rank, exactly as in the tutorial."""
    df = df.assign(rank=df["concept:name"].map(rank).fillna(99))
    df = df.sort_values(["case:concept:name", "time:timestamp", "rank"], kind="stable").drop(columns="rank")
    df["@@index"] = range(len(df))   # PM4Py uses this column to order events with equal timestamps
    return df

def reorder_events(df):
    return reorder(df, RANK)

GPS_OF = {"Accept GPS fix": ACCEPT, "Pickup GPS fix": PICKUP}

def merge_gps_fixes(df):
    """Abstraction (event aggregation): a GPS fix in the same minute as its event adds nothing, so it is
    merged into that event. A fix *before* the event stays: it shows a lag between location fix and record."""
    ref = df[df["concept:name"].isin(GPS_OF.values())][["case:concept:name", "concept:name", "time:timestamp"]]
    ref = ref.set_axis(["case:concept:name", "ref", "ref_ts"], axis=1)
    m = df[["case:concept:name"]].assign(ref=df["concept:name"].map(GPS_OF)).merge(ref, how="left", on=["case:concept:name", "ref"])
    same_minute = m["ref_ts"].to_numpy() == df["time:timestamp"].to_numpy()
    return df[~same_minute]

def add_case_attributes(df):
    """Enrichment: plan vs reality per parcel, needed for the analysis question."""
    t = df.groupby(["case:concept:name", "concept:name"])["time:timestamp"].first().unstack()
    wait = (t[PICKUP] - t[ACCEPT]).dt.total_seconds() / 60
    status = pd.Series("on time", index=t.index)
    status[t[PICKUP] < t["Window opens"]] = "early"
    status[t[PICKUP] > t["Window closes"]] = "late"
    attrs = pd.DataFrame({
        "case:pickup_status": status,
        "case:wait_min": wait,
        "case:long_wait": wait > wait.quantile(0.9),
        "case:window_h": (t["Window closes"] - t["Window opens"]).dt.total_seconds() / 3600,
    })
    return df.merge(attrs, left_on="case:concept:name", right_index=True, how="left")

### 5. Equal timestamps
Timestamps have minute resolution, so many events of a parcel share a minute (for example, a pickup and its GPS fix, or an accept at the moment the window opens).

In [6]:
ties = log_raw.duplicated(["case:concept:name", "time:timestamp"], keep=False)
print(f"{ties.mean():.0%} of the events share their timestamp with another event of the same parcel")

45% of the events share their timestamp with another event of the same parcel


### 6. Running the pipeline

In [7]:
df = log_raw
stages = {"1 raw": log_raw}
for i, step in enumerate([remove_duplicates, keep_complete, drop_pickup_before_accept,
                          reorder_events, merge_gps_fixes, add_case_attributes], 1):
    df = step(df)
    steps.append(log_summary(df, f"{i} {step.__name__}"))
    if step.__name__ in ("drop_pickup_before_accept", "reorder_events", "merge_gps_fixes"):
        stages[{"drop_pickup_before_accept": "2 filtered", "reorder_events": "3 reordered",
                "merge_gps_fixes": "4 abstracted"}[step.__name__]] = df
parcel_log = df
pd.DataFrame(steps).to_csv(TAB / "02_preprocessing_steps.csv", index=False)

1 remove_duplicates          cases=261,801 events=1,446,921 activities= 6 variants=   65


2 keep_complete              cases=261,801 events=1,446,921 activities= 6 variants=   65


3 drop_pickup_before_accept  cases=261,801 events=1,446,921 activities= 6 variants=   65


4 reorder_events             cases=261,801 events=1,446,921 activities= 6 variants=   49


5 merge_gps_fixes            cases=261,801 events=1,151,832 activities= 6 variants=   27


6 add_case_attributes        cases=261,801 events=1,151,832 activities= 6 variants=   27


**Effect of each step**
- The three filters remove nothing in Jilin: no duplicates, every parcel has an accept and a pickup, no pickup before accept. They stay in the pipeline because the other four cities may differ.
- **Reordering** removes the artificial variants caused by file order at equal timestamps (Jilin: 65 → 49 variants).
- **Merging GPS fixes** removes the fixes that only repeat their event (Jilin: 295,089 events, 49 → 27 variants); the remaining fixes are the ones taken before the event was recorded.
- **Enrichment** adds the case attributes for the analysis question; it does not change the control flow.

### 7. Measuring the effect with FEEED
As in the tutorial, we export each stage to XES and extract log features. Two deviations:
- FEEED's entropy features need Java, which is not part of our setup, so they are left out.
- Writing four full XES files (about 1.3 million events each) takes long, so the features are computed on the same **random 10% of parcels** at every stage.

In [8]:
from feeed.feature_extractor import extract_features

# shortcut: features on a fixed 10% case sample; rerun on all cases for the final log (M4)
sample = pd.Series(parcel_log["case:concept:name"].unique()).sample(frac=0.1, random_state=0)
feature_types = ["simple_stats", "trace_variant", "activities", "dfg_based"]
tmp_dir, results = tempfile.mkdtemp(), {}
for name, stage in stages.items():
    path = os.path.join(tmp_dir, name.replace(" ", "_") + ".xes")
    pm4py.write_xes(stage[stage["case:concept:name"].isin(sample)], path)
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        results[name] = extract_features(path, feature_types)
features = pd.DataFrame(results).drop(index="log").astype(float)
features.to_csv(TAB / "02_feeed_features.csv")
selected = ["n_traces", "n_variants", "ratio_variants_per_number_of_traces", "ratio_top_10_variants",
            "n_unique_activities", "density_dfg"]
features.loc[[f for f in selected if f in features.index]].round(3)

,1 raw,2 filtered,3 reordered,4 abstracted
n_traces,26180.000,26180.000,26180.000,26180.000
n_variants,52.000,52.000,42.000,22.000
ratio_variants_per_number_of_traces,0.002,0.002,0.002,0.001
ratio_top_10_variants,0.607,0.607,0.585,0.639
n_unique_activities,6.000,6.000,6.000,6.000
density_dfg,0.554,0.554,0.518,0.429


## Part C · Making sense of the cleaned log

### 8. Variants: what do the stories say?

In [9]:
variants = (parcel_log.groupby("case:concept:name")["concept:name"].agg(" → ".join)
            .value_counts().rename("parcels").to_frame())
variants["share %"] = (variants.parcels / variants.parcels.sum() * 100).round(1)
variants["cumulative %"] = variants["share %"].cumsum().round(1)
variants.head(10).to_csv(TAB / "02_variants_top10.csv")
save_dfg_percent(parcel_log, FIG / "02_dfg_parcel_clean.png", "Parcel log, cleaned (event log v1)", unit="parcels")
print(f"{len(variants)} variants; the 3 most frequent cover {variants['share %'].head(3).sum():.1f}% of the parcels")
variants.head(10)

27 variants; the 3 most frequent cover 77.8% of the parcels


,parcels,share %,cumulative %
concept:name,,,
Accept order → Window opens → Pick up order → Window closes,119132,45.5,45.5
Accept order → Pick up order → Window opens → Window closes,48500,18.5,64.0
Accept GPS fix → Accept order → Window opens → Pick up order → Window closes,36224,13.8,77.8
Accept order → Window opens → Pickup GPS fix → Pick up order → Window closes,20663,7.9,85.7
Accept GPS fix → Accept order → Pick up order → Window opens → Window closes,15246,5.8,91.5
Accept order → Pickup GPS fix → Pick up order → Window opens → Window closes,8650,3.3,94.8
Accept GPS fix → Accept order → Window opens → Pickup GPS fix → Pick up order → Window closes,7838,3.0,97.8
Accept GPS fix → Accept order → Pickup GPS fix → Pick up order → Window opens → Window closes,3385,1.3,99.1
Window opens → Accept order → Pick up order → Window closes,726,0.3,99.4


In [10]:
# Read each variant as plan vs reality: status and waiting time per variant (top 10)
case_view = parcel_log.groupby("case:concept:name").agg(
    variant=("concept:name", " → ".join), status=("case:pickup_status", "first"),
    wait_min=("case:wait_min", "first"), long_wait=("case:long_wait", "first"))
top = variants.index[:10]
(case_view[case_view.variant.isin(top)]
 .groupby("variant").agg(parcels=("status", "size"), status=("status", lambda s: s.mode()[0]),
                          median_wait_min=("wait_min", "median"), long_wait_pct=("long_wait", "mean"))
 .assign(long_wait_pct=lambda d: (d.long_wait_pct * 100).round(1))
 .sort_values("parcels", ascending=False))

,parcels,status,median_wait_min,long_wait_pct
variant,,,,
Accept order → Window opens → Pick up order → Window closes,119132,on time,110.0,10.8
Accept order → Pick up order → Window opens → Window closes,48500,early,47.0,11.2
Accept GPS fix → Accept order → Window opens → Pick up order → Window closes,36224,on time,104.0,7.5
Accept order → Window opens → Pickup GPS fix → Pick up order → Window closes,20663,on time,113.0,11.4
Accept GPS fix → Accept order → Pick up order → Window opens → Window closes,15246,early,42.0,5.5
Accept order → Pickup GPS fix → Pick up order → Window opens → Window closes,8650,early,48.0,11.8
Accept GPS fix → Accept order → Window opens → Pickup GPS fix → Pick up order → Window closes,7838,on time,103.0,7.2
Accept GPS fix → Accept order → Pickup GPS fix → Pick up order → Window opens → Window closes,3385,early,41.0,4.5
Window opens → Accept order → Pick up order → Window closes,726,on time,33.0,0.4


How to read the variants:
- *Window opens* before *Pick up order* before *Window closes*: on time. *Pick up order* before *Window opens*: **early**; after *Window closes*: **late**.
- *Window opens* before *Accept order*: the task was accepted after its window had already started.
- A remaining **GPS fix before its event**: the courier's location was fixed before the accept or pickup was recorded in the app (a recording lag).
- Parcels without a GPS fix event either had the fix in the same minute (merged) or no fix at all (missing in the raw data).

### 9. Second view: courier-day (batching, divergent by construction)
The same cleaned events with the courier-day as case, on the two courier activities only. This is the view for batching; its loops are partly caused by divergence (section 3). Notebook 01 shows its DFG and BPMN.

In [11]:
courier_day_log = pm4py.format_dataframe(
    parcel_log[parcel_log["concept:name"].isin([ACCEPT, PICKUP])]
    .drop(columns=["case:concept:name", "@@index", "@@case_index"], errors="ignore")
    .rename(columns={"courier_day": "cd"}).reset_index(drop=True),
    case_id="cd", activity_key="concept:name", timestamp_key="time:timestamp")
courier_day_log = reorder(courier_day_log, RANK)
_ = log_summary(courier_day_log, "courier-day view")

courier-day view             cases= 18,880 events=  523,602 activities= 2 variants=9,567


### 10. Export: event log v1
The cleaned parcel log is written as compressed XES (not committed; it is rebuilt by running this notebook) and read back to check that nothing is lost.

In [12]:
xes = PROC / "lade_jl_pickup_parcel_v1.xes.gz"
pm4py.write_xes(parcel_log.drop(columns=["courier_day"]), str(xes))
back = pm4py.read_xes(str(xes))
assert len(back) == len(parcel_log) and back["case:concept:name"].nunique() == parcel_log["case:concept:name"].nunique()
print(f"{xes.name}: {xes.stat().st_size / 1e6:.1f} MB, {len(back):,} events, {back['case:concept:name'].nunique():,} cases")

lade_jl_pickup_parcel_v1.xes.gz: 27.4 MB, 1,151,832 events, 261,801 cases


## Open points for the next consultation
- Is the parcel log with six activities the right main log, with the courier-day only as a batching view?
- GPS fixes before their event: do we treat them as a recording lag (as now), or should the GPS time replace the recorded time?
- Should the long-window parcels (window of about 15 hours) be kept in the same log or analysed separately?
- Pickup rounds on the courier level (Start/End round) as a separate courier log for the enhancement part?